# Vector Embeddings

Compute vector embeddings of segments and store them.

In [ ]:
import { readFileSync } from 'node:fs';

import { SystemMessage, HumanMessage } from "@langchain/core/messages";
import { ChatPromptTemplate, MessagesPlaceholder } from "@langchain/core/prompts";
import { JsonOutputParser } from "@langchain/core/output_parsers";
import { CharacterTextSplitter } from "langchain/text_splitter";
import { OpenAIEmbeddings } from "@langchain/openai";
import { MemoryVectorStore } from "langchain/vectorstores/memory";

import { ROOT_DIR } from '../../server/src/util/fileUtils.ts';
import { getNotebookLogger } from '../../server/src/Logger.ts';
import { newModel } from '../../server/src/agents/agent.ts';

const lhsText = readFileSync(`${ROOT_DIR}/experiments/20250409-latex-to-markdown/selected-AES.md`, 'utf-8');

const segments = lhsText.split('\n\n');
const splitter = new CharacterTextSplitter();
const splittedDocs = await splitter.createDocuments(segments);
console.log("Number of segments", segments.length);
console.log("Number of documents", splittedDocs.length);

const embeddings = new OpenAIEmbeddings({
  model: "text-embedding-3-large"
});
const vectorStore = new MemoryVectorStore(embeddings);
await vectorStore.addDocuments(splittedDocs);

const numResults = 5;
const query = segments[13];
const searchResults = await vectorStore.similaritySearchWithScore(query, numResults);

console.log("Query:", query);
console.log("Search results:");
for (const [doc, score] of searchResults) {
  console.log(`* [SIM=${score.toFixed(3)}] ${doc.pageContent}`);
}


Number of segments 62
Number of documents 62


Query: The four transformations are specified in Sections 5.1.1-5.1.4. In those specifications, the transformed bit, byte, or block is denoted by appending the symbol ' as a superscript on the original variable (i.e., by $b_{i}^{\prime}$, $b^{\prime}$, $s_{i, j}^{\prime}$, or $s^{\prime}$).
Search results:
* [SIM=1.000] The four transformations are specified in Sections 5.1.1-5.1.4. In those specifications, the transformed bit, byte, or block is denoted by appending the symbol ' as a superscript on the original variable (i.e., by $b_{i}^{\prime}$, $b^{\prime}$, $s_{i, j}^{\prime}$, or $s^{\prime}$).
* [SIM=0.656] 2. Apply the following affine transformation of the bits of $\tilde{b}$ to produce the bits of $b^{\prime}$:
$$
b_{i}^{\prime}=\tilde{b}_{i} \oplus \tilde{b}_{(i+4) \bmod 8} \oplus \tilde{b}_{(i+5) \bmod 8} \oplus \tilde{b}_{(i+6) \bmod 8} \oplus \tilde{b}_{(i+7) \bmod 8} \oplus c_{i}
$$
* [SIM=0.624] The action of this transformation is illustrated in Fig. 5, where $l=4*$ rou